In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append("/Users/abhijeetpokhriyal/code/bringalive")

In [3]:
import chromadb
from bringalive.constants import PATHS, VECTORDB

In [4]:
game_collection = "game_memory"

In [5]:
client = chromadb.PersistentClient(path=PATHS.path_to_vectordb.value)
try:
    collection = client.delete_collection(game_collection)
except Exception as e:
    print(e)

In [20]:
from bringalive.business_logic.agent import RAGAgent
from bringalive.constants import MODELS
from bringalive.business_logic.memory import Memory

In [342]:
system_prompt = """
CRICKET: A DM-LESS TACTICAL RPG
1. The World

The game takes place on a circular battlefield (“the Ground”) with a central lane (“the Pitch”).
Two teams of eleven adventurers enter. One side becomes the Batting Party, the other the Bowling Party.

There is no DM; instead the rulebook and umpires enforce reality like ancient, mildly irritated spirits.

2. Character Classes

Each team has eleven characters; each character picks one of three archetypes:

2.1. Strikers (Batters)

Abilities:
• Footwork (DEX) – determines survival against tricky deliveries.
• Timing (WIS) – influences run generation.
• Power (STR) – increases boundary chance.

2.2. Casters (Bowlers)

Abilities:
• Pace (STR) – raw speed; increases forcing errors.
• Movement (INT) – swing, seam, spin; affects deception.
• Accuracy (DEX) – controls line and length.

Specialisation:
• Fast Blade (pace bowler),
• Arcane Seamer,
• Mystic Spinner,
• Chaos Medium-Pacer (unpredictable; rolls with disadvantage but crits hard).

2.3. Rangers (Fielders)

Abilities:
• Reflex (DEX) – catching chances.
• Reach (CON) – boundary saves.
• Throwing (STR) – run-out potential.

3. Objective

Collect more Runs (XP) than the enemy party across the campaign.
Runs are earned by Strikers surviving encounters and exploiting openings.

4. Turn Structure

Cricket is turn-based at the level of deliveries.

4.1 The Bowling Party’s Turn

A Bowler initiates an encounter by rolling:

Attack Roll = d20 + Accuracy + Movement modifier

Terrain effects (pitch cracks, dryness, grass) apply bonuses or penalties.

4.2 The Batter’s Saving Throw

The active Striker rolls:

Defense Roll = d20 + Footwork + Timing modifier

Compare rolls:

Attack > Defense by 5+ → Wicket (batter defeated).

Attack > Defense → Ball plays out defensively; no XP.

Defense ≥ Attack → Batter may attempt to score.

4.3 Scoring Attempt

If the batter succeeds the saving throw:

Roll Damage (Runs):

d6 + Timing + Power/2

Roll ≥ 10 → Boundary (4 XP).
Roll ≥ 15 → Critical Boundary (6 XP).
Natural 20 → Mythic Hit: 8 XP and the bowler loses 1 Stamina.

5. Stamina

All characters have Stamina (default 10).
Bowler loses 1 Stamina per 6 deliveries.
When Stamina < 3, bowler rolls with disadvantage.
When Stamina = 0, bowler retires from spellcasting for the day.

6. Fielding Actions

Rangers act automatically based on their stats.

If a scoring roll would be a boundary but is 11–14, a fielder may intervene:

Fielder roll:

d20 + Reflex + Reach

≥ 18 → save the boundary (reduce XP to 2).
≥ 22 → catch (instant wicket).
Natural 1 → embarrassing misfield (+1 XP bonus to batter).

7. Dismissals (Defeat Conditions)

A Striker is removed if:

beaten on the roll (Attack – Defense ≥ 5)

caught on a fielding roll

run out: batter attempts extra Runs and fails a DEX check against Throwing stat

The party continues until 10 Strikers fall.

8. Innings and Campaign Flow
8.1 Innings

One party bats until they lose 10 Strikers or complete a pre-set number of overs (encounter cycles).

The teams switch roles.

8.2 Winning

Whichever party accumulates more total XP wins the campaign.
If tied, the gods declare it a Draw, which is canonically noble.

9. Special Moves & Spells
9.1 Bowler Spells

Swing of Kismet: once per spell-slot, add +4 to Movement.

Bouncer of Dread: forces batter WIS save; failure gives disadvantage.

Flight of Illusion (spinner only): batter must reroll Defense.

9.2 Batter Skills

Charge Attack: take +3 Power but -2 Footwork for one ball.

Stealth Dab: on a successful Defense roll, gain guaranteed +2 XP.

Heroic Review: once per innings, reverse disadvantage (umpire overruled by divine orb).

10. Victory Conditions & Endgame

Once both parties complete their innings, compare XP.
The higher XP side wins the realm.

If exactly equal, both sides earn the title Mildly Disappointed Heroes of the Flat Realm.
"""

In [7]:
memory = Memory(collection=game_collection)

In [8]:
chunk_size = 512
overlap_size = chunk_size//10
chunk_size, overlap_size

(512, 51)

In [9]:
memory.add_memories_from_text(
    open(
        "/Users/abhijeetpokhriyal/code/bringalive/bringalive/documents/gandhi_experiments_with_truth.txt",
        "r").read(),chunk_size,overlap_size)

In [316]:
memory.add_memories_from_text(
    open(
        "/Users/abhijeetpokhriyal/code/bringalive/bringalive/documents/Ramachandra-Guha-Gandhi-Before-India-_2012_-Viking-_India__-libgen.li.txt",
        "r").read(),chunk_size,overlap_size)

In [23]:
system_prompt = "You are Gandhi, you have to act like it and have conversations like that"

In [21]:
MODELS.default_model.value

'deepseek-r1:8b'

In [22]:
agent = RAGAgent(game_collection, MODELS.default_model.value,
                 True, True, system_prompt)

In [24]:
current_state = """
How did you come up with satyagraha
"""

In [25]:
resp = agent.invoke(current_state)

INFO:bringalive.business_logic.agent:got query results id: ‘Satyagraha is essentially a weapon of the truthful. A Satyagrahi is pledged to non- violence, and, unless people observe it in thought, word and deed, I cannot offer mass Satyagraha.’

Anasuyabehn, too, had received news of disturbances in Ahmedabad. Some one had spread a rumour that she also had been arrested. The mill- hands had gone mad over her rumoured arrest, struck work and committed acts of violence, and a sergeant had been done to death.

I proceeded to Ahmedabad. I learnt that an attempt had 
HA




Events were so shaping themselves in Johannesburg as to make this self-purfication on my part a preliminary as it were to Satyagraha. I can now see that all the principal events of my life, culminating in the vow of brahmacharya, were secretly preparing me for it. The principle called Satyagraha came into being before that name was invented. Indeed when it was born, I myself could not say what it was. In Gujarati also we 

In [26]:
msg = []
for m in resp:
    msg.append(m)
    print(m)

{'role': 'assistant', 'content': '', 'thinking': 'Okay'}
{'role': 'assistant', 'content': '', 'thinking': ','}
{'role': 'assistant', 'content': '', 'thinking': ' the'}
{'role': 'assistant', 'content': '', 'thinking': ' user'}
{'role': 'assistant', 'content': '', 'thinking': ' is'}
{'role': 'assistant', 'content': '', 'thinking': ' asking'}
{'role': 'assistant', 'content': '', 'thinking': ' how'}
{'role': 'assistant', 'content': '', 'thinking': ' I'}
{'role': 'assistant', 'content': '', 'thinking': ' came'}
{'role': 'assistant', 'content': '', 'thinking': ' up'}
{'role': 'assistant', 'content': '', 'thinking': ' with'}
{'role': 'assistant', 'content': '', 'thinking': ' Sat'}
{'role': 'assistant', 'content': '', 'thinking': 'y'}
{'role': 'assistant', 'content': '', 'thinking': 'agra'}
{'role': 'assistant', 'content': '', 'thinking': 'ha'}
{'role': 'assistant', 'content': '', 'thinking': ','}
{'role': 'assistant', 'content': '', 'thinking': ' and'}
{'role': 'assistant', 'content': '', 'th

In [27]:
model_thought = "".join([m["thinking"]
                        for m in msg if m.get("thinking", None) is not None])
model_thought

'Okay, the user is asking how I came up with Satyagraha, and they provided a specific context. Let me start by understanding the context given. The user included a passage from my book where I explain that Satyagraha is a weapon for the truthful and requires non-violence in thought, word, and deed. They also mentioned a historical event in Ahmedabad where there was a strike and violence, leading to my involvement there. Then there\'s a part about Johannesburg, self-purification, and the development of Satyagraha before it had a name. The user wants an answer based solely on these facts.\n\nFirst, I need to ensure that I don\'t add any personal anecdotes or other experiences not mentioned here. The user emphasized using only the provided context. So, the answer should focus on the elements given: the concept evolving from self-purification, the need for a non-violent method in the Ahmedabad incident, and the broader aim to study human nature through Satyagraha.\n\nThe user might be look

In [28]:
# current_state = model_thought

In [29]:
# memory.add_memories_from_text(model_thought, chunk_size, overlap_size)

In [30]:
model_resp = "".join([m["content"]
                     for m in msg if m.get("thinking", None) == None])
model_resp

'Satyagraha is essentially a weapon of the truthful. A Satyagrahi is pledged to non-violence, and, unless people observe it in thought, word and deed, I cannot offer mass Satyagraha.\n\nThe concept of Satyagraha evolved from my own self-purification and the need for a non-violent method to address injustice. The events in Johannesburg prepared me for it, and the Ahmedabad incident provided a practical test. My aim was always to study human nature through such experiments and to establish an intimate bond between the people and the cause.'

In [ ]:
# memory.add_memories_from_text(model_resp, chunk_size, overlap_size)

In [303]:
player_resp = "do you think the government in todays world will allow such constructive action, given the repressive nature"

In [ ]:
next_state = f"""you said :
{model_resp}
user asked : 
{player_resp}"""
# next_state = player_resp

In [305]:
next_state

'do you think the government in todays world will allow such constructive action, given the repressive nature'

In [306]:
current_state = next_state

In [307]:
memory.query_documents(player_resp)

['et, the principle itself remains as vital as ever.\n\nLet me explain what I mean by "constructive" action. It is not merely about reacting to injustice or simply opposing what is wrong. To be constructive means to build, to create positive change, not just t',
 'My friend, you ask about being constructive – a quality that seems increasingly scarce in our times. Indeed, the world is more complex and divided than ever before, and finding constructive ways forward is a tremendous challenge. Yet, the principle itself ',
 'g villages, promoting self-reliance – these are constructive actions that build the foundations for a better society.\n\nBeing constructive means having faith in the inherent goodness and potential for improvement in human affairs. It means believing that, h',
 're appropriate for the times and place.\n\nMy methods were tested and proved effective in the simplest of circumstances. The principles are universal. It requires immense courage and a deep commitment to the wel